--- 
# **[실습]**

### 실습 목표

Contextual Retrieval을 실제 문서에 적용하여 검색 성능을 개선합니다.

### 난이도별 가이드

**기본 난이도:**
- 제공된 샘플 문서에 Contextual Retrieval 적용
- 일반 검색 vs Contextual 검색 성능 비교
- 3개 이상의 쿼리로 테스트

**중급 난이도:**
- 자체 문서(예: 기술 문서, 위키피디아) 활용
- Hybrid 검색 (Embedding + BM25) 구현
- 가중치 조합 실험 (0.3:0.7, 0.5:0.5, 0.7:0.3)

**고급 난이도:**
- 다국어 문서에 Contextual Retrieval 적용
- Reranker와 결합하여 최적 파이프라인 구성
- 검색 성능 지표 (HitRate, MRR) 측정 및 비교

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from pprint import pprint
from typing import List, Tuple

In [3]:
from langfuse.langchain import CallbackHandler

# LangChain 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

`(1) 문서 준비 및 청킹`

자신의 문서를 로드하고 청크로 분할합니다.

In [10]:
import os
import re
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

# 1. 파일 경로 및 로더 지정
file_path = "./data/industry_report/"

loader = DirectoryLoader(
    file_path,
    glob="*.pdf",           # 하위 폴더 미포함 / 포함 : **/*.pdf
    loader_cls=PyMuPDFLoader  
)

# 2. 문서 로드
docs = loader.load()

# ==========================================
# 3. 파일명 기반 메타데이터 동적 추출 및 할당 (추가된 부분)
# ==========================================
# 정규표현식 패턴: [기업명]보고서종류(YYYY.MM.DD).pdf
# 설명: \[ (대괄호 열기) / (.*?) (아무글자나 매칭) / \] (대괄호 닫기)
pattern = r"\[(.*?)\](.*?)\((.*?)\)\.pdf"

for doc in docs:
    # 전체 경로에서 파일명만 추출 (예: ./data/industry_report/[삼성전자]분기보고서(2026.05.15).pdf -> [삼성전자]분기보고서(2026.05.15).pdf)
    source_path = doc.metadata.get('source', '')
    file_name = os.path.basename(source_path)
    
    # 정규식 패턴과 파일명 매칭
    match = re.match(pattern, file_name)
    
    if match:
        company_name = match.group(1)  # 삼성전자
        report_type = match.group(2)   # 분기보고서
        report_date = match.group(3)   # 2026.05.15
        
        # Document 객체의 metadata 딕셔너리에 추출한 값 주입
        doc.metadata['company'] = company_name
        doc.metadata['report_type'] = report_type
        doc.metadata['date'] = report_date
        
        # (선택 팁) 날짜에서 '연도'만 분리해서 메타데이터로 빼두면, 
        # 나중에 "2026년도 리포트만 찾아줘" 할 때 DB 필터링이 훨씬 수월합니다.
        if "." in report_date:
            doc.metadata['year'] = report_date.split('.')[0] 
            
    else:
        # 패턴에 맞지 않는 예외 파일이 있을 경우의 처리
        doc.metadata['company'] = "Unknown"
        doc.metadata['report_type'] = "Unknown"
        print(f"경고: 파일명 패턴 불일치 (메타데이터 추출 실패) -> {file_name}")

# ==========================================
# 4. 로드 및 메타데이터 주입 결과 확인
# ==========================================
print("="*80)
print(f"로드된 총 페이지(Document) 수: {len(docs)}")
print("="*80)

if docs:
    print(f"첫 번째 페이지 출처 (원본): {docs[0].metadata['source']}")
    print("-" * 40)
    print(f"✅ 주입된 메타데이터 정보:\n{docs[0].metadata}")
    print("-" * 40)
    print(f"첫 번째 페이지 내용 일부:\n{docs[0].page_content[:200]}...")

로드된 총 페이지(Document) 수: 323
첫 번째 페이지 출처 (원본): data\industry_report\[삼성전자]분기보고서(2026.05.15).pdf
----------------------------------------
✅ 주입된 메타데이터 정보:
{'producer': 'iText® 5.4.0 ©2000-2012 1T3XT BVBA (AGPL-version)', 'creator': '', 'creationdate': '2026-05-15T16:14:23+09:00', 'source': 'data\\industry_report\\[삼성전자]분기보고서(2026.05.15).pdf', 'file_path': 'data\\industry_report\\[삼성전자]분기보고서(2026.05.15).pdf', 'total_pages': 323, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-05-15T16:14:23+09:00', 'trapped': '', 'modDate': "D:20260515161423+09'00'", 'creationDate': "D:20260515161423+09'00'", 'page': 0, 'company': '삼성전자', 'report_type': '분기보고서', 'date': '2026.05.15', 'year': '2026'}
----------------------------------------
첫 번째 페이지 내용 일부:
목                 차
분 기 보 고 서............................................................................................................................................1
【 대표이사 등의 확인 】..................


In [16]:
MAX_PAGES = 22

# PyMuPDFLoader는 메타데이터의 'page' 값을 0부터 시작합니다 (0 = 1페이지).
# 따라서 page 값이 22 미만(0~21)인 문서만 추려냅니다.
filtered_docs = [doc for doc in docs if doc.metadata.get('page', 0) < MAX_PAGES]

print(f"전체 로드된 페이지: {len(docs)}장 -> {MAX_PAGES}페이지 이하 필터링 후: {len(filtered_docs)}장")


# ==========================================
# [수정] 목차(TOC)와 본문 동적 분리 로직
# ==========================================
toc_docs = []
body_docs = []

# 정규식 패턴: '점선이나 넓은 공백 뒤에 숫자가 오는 패턴'을 찾습니다. 
toc_pattern = re.compile(r'(?:\.{3,}|\s{4,})\d+') 

# 기존 docs 대신 필터링된 filtered_docs를 사용합니다!
for doc in filtered_docs:
    content = doc.page_content
    
    # 1. 명시적 키워드 확인 (페이지 상단 200자 이내에 키워드가 있는지)
    header_text = content[:200].lower()
    has_toc_keyword = "목차" in header_text or "contents" in header_text or "index" in header_text
    
    # 2. 정규식 패턴 확인 (목차 패턴이 해당 페이지에 3번 이상 반복되는가?)
    pattern_matches = len(toc_pattern.findall(content))
    
    # 판단 로직: 목차 키워드가 있거나, 목차 패턴이 확연히 나타나면 TOC로 분류
    if has_toc_keyword or pattern_matches >= 3:
        doc.metadata['is_toc'] = True
        toc_docs.append(doc)
    else:
        doc.metadata['is_toc'] = False
        body_docs.append(doc)

# 향후 Contextual Retrieval을 위해 목차 텍스트들만 따로 묶어서 딕셔너리로 보관
toc_text_map = {}
for doc in toc_docs:
    filename = doc.metadata.get('source', 'unknown')
    if filename not in toc_text_map:
        toc_text_map[filename] = ""
    toc_text_map[filename] += doc.page_content + "\n"

# ==========================================
# 4. 분리 결과 확인 및 청킹
# ==========================================
print(f"✅ 분류 완료: 총 {len(filtered_docs)}페이지 중 목차 {len(toc_docs)}페이지, 본문 {len(body_docs)}페이지")



전체 로드된 페이지: 323장 -> 22페이지 이하 필터링 후: 22장
✅ 분류 완료: 총 22페이지 중 목차 3페이지, 본문 19페이지


In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # 작은 청크로 분할
    chunk_overlap=50,    # 50자 중복
    separators=["\n\n", "\n", ". ",],
)

# 문서 분할
chunks = text_splitter.split_documents(body_docs)

print(f"생성된 청크 수: {len(chunks)}")
print("="*80)

for i, chunk in enumerate(chunks[:5]):
    print(f"\n[청크 {i+1}] ({len(chunk.page_content)}자)")
    print("-"*40)
    print(chunk.page_content[:200] + "..." if len(chunk.page_content) > 200 else chunk.page_content)

생성된 청크 수: 37

[청크 1] (433자)
----------------------------------------
분 기 보 고 서
 
 
 
 
                                    (제 58 기) 
 
사업연도
2026.01.01
부터
2026.03.31
까지
금융위원회
한국거래소 귀중
2026년     5월     15일
제출대상법인 유형 :
주권상장법인
면제사유발생 :
해당사항 없음
회      사      명 :
삼성전자주식회사
대 ...

[청크 2] (44자)
----------------------------------------
【 대표이사 등의 확인 】
전자공시시스템 dart.fss.or.kr
Page 2

[청크 3] (424자)
----------------------------------------
I. 회사의 개요
1. 회사의 개요
 
회사의 개요는 기업공시서식 작성기준에 따라 분기보고서에 기재하지 않습니다.
(반기ㆍ사업보고서에 기재 예정) 
 
2. 회사의 연혁
 
회사의 연혁은 기업공시서식 작성기준에 따라 분기보고서에 기재하지 않습니다.
(사업보고서 등에 기재 예정)
 
3. 자본금 변동사항
 
자본금 변동사항은 기업공시서식 작성기준에 따라 분기...

[청크 4] (462자)
----------------------------------------
II. 사업의 내용
 
1. 사업의 개요
 
당사는 본사를 거점으로 한국과 DX 부문 산하 해외 9개 지역총괄 및 DS 부문 산하 해외 5개
지역총괄의 생산ㆍ판매법인, SDC 및 Harman 산하 종속기업 등 310개의 종속기업으로 구성
된 글로벌 전자 기업입니다.
 
사업별로 보면, Set 사업은 DX(Device eXperience) 부문이 TV를 비롯...

[청크 5] (485자)
----------------------------------------
바 스피커 등 컨슈머오디오 제품 등을 개발, 생산, 판매하고 있습니다.
 
☞ 부문별 사업에

`(2) 컨텍스트 생성`

각 청크에 대해 맥락 설명을 생성합니다.

In [18]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# LLM 초기화 (컨텍스트 생성용 - 가벼운 모델 사용)
context_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# Anthropic의 컨텍스트 생성 프롬프트 (한국어 버전)
context_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 문서의 청크에 맥락을 추가하는 전문가입니다.
주어진 청크가 전체 문서에서 어떤 위치에 있고 무엇에 대한 내용인지 간결하게 설명하세요.
설명은 50-100자 이내로 작성하세요.
오직 맥락 설명만 출력하세요."""),
    ("user", """<document>
{whole_document}
</document>

위 문서에서 아래 청크의 맥락을 설명해주세요:

<chunk>
{chunk_content}
</chunk>

맥락 설명:""")
])

# 컨텍스트 생성 체인
context_chain = context_prompt | context_llm | StrOutputParser()

print("컨텍스트 생성 체인 준비 완료")

컨텍스트 생성 체인 준비 완료


In [19]:
from langchain_core.documents import Document

# 각 청크에 대해 컨텍스트 생성
def generate_contexts(chunks: List[Document], whole_document: str) -> List[str]:
    """청크들에 대한 컨텍스트를 배치로 생성합니다."""
    inputs = [
        {"whole_document": whole_document, "chunk_content": chunk.page_content}
        for chunk in chunks
    ]
    
    # 배치 처리로 효율적으로 생성
    contexts = context_chain.batch(
        inputs, 
        #config={"max_concurrency": 5, "callbacks": [langfuse_handler]}
    )
    
    return contexts

# 컨텍스트 생성 실행
print("컨텍스트 생성 중...")
contexts = generate_contexts(chunks, body_docs)

print(f"\n생성된 컨텍스트 수: {len(contexts)}")
print("="*80)

for i, (chunk, context) in enumerate(zip(chunks[:5], contexts[:5])):
    print(f"\n[청크 {i+1}]")
    print(f"원본: {chunk.page_content[:100]}...")
    print(f"컨텍스트: {context}")

컨텍스트 생성 중...

생성된 컨텍스트 수: 37

[청크 1]
원본: 분 기 보 고 서
 
 
 
 
                                    (제 58 기) 
 
사업연도
2026.01.01
부터
2026.03.31
까지
금...
컨텍스트: 삼성전자 2026년 1분기 분기보고서 표지로, 회사 기본정보와 제출일 등이 포함된 문서 첫 페이지입니다.

[청크 2]
원본: 【 대표이사 등의 확인 】
전자공시시스템 dart.fss.or.kr
Page 2...
컨텍스트: 분기보고서 초반부에 대표이사 등의 확인 서명 또는 인증 내용을 간략히 표기한 페이지입니다.

[청크 3]
원본: I. 회사의 개요
1. 회사의 개요
 
회사의 개요는 기업공시서식 작성기준에 따라 분기보고서에 기재하지 않습니다.
(반기ㆍ사업보고서에 기재 예정) 
 
2. 회사의 연혁
 
회사의...
컨텍스트: 분기보고서 초반부로, 회사 개요 및 주요 정보는 분기보고서에 미기재하고 반기·사업보고서에 기재 예정임을 안내하는 내용입니다.

[청크 4]
원본: II. 사업의 내용
 
1. 사업의 개요
 
당사는 본사를 거점으로 한국과 DX 부문 산하 해외 9개 지역총괄 및 DS 부문 산하 해외 5개
지역총괄의 생산ㆍ판매법인, SDC 및 ...
컨텍스트: 삼성전자 분기보고서 중 사업 내용 소개 부분으로, 회사의 글로벌 조직과 주요 사업 부문별 제품 현황을 설명함.

[청크 5]
원본: 바 스피커 등 컨슈머오디오 제품 등을 개발, 생산, 판매하고 있습니다.
 
☞ 부문별 사업에 관한 자세한 사항은 '7. 기타 참고사항'의 '다. 사업부문별 현황'과   '라. 사
...
컨텍스트: 삼성전자의 사업부문별 개요와 국내외 종속기업 현황을 소개하는 2026년 1분기 분기보고서 초반부 내용입니다.


In [20]:
# 컨텍스트가 추가된 청크 생성
contextual_chunks = []

for i, (chunk, context) in enumerate(zip(chunks, contexts)):
    # 컨텍스트 + 원본 청크 결합
    contextual_content = f"[맥락] {context}\n\n{chunk.page_content}"
    
    contextual_chunk = Document(
        page_content=contextual_content,
        metadata={
            **chunk.metadata,
            "chunk_id": i,
            "original_content": chunk.page_content,
            "context": context,
        }
    )
    contextual_chunks.append(contextual_chunk)

print(f"Contextual 청크 생성 완료: {len(contextual_chunks)}개")
print("="*80)

# 예시 출력
print("\n[Contextual 청크 예시]")
print(contextual_chunks[3].page_content)

Contextual 청크 생성 완료: 37개

[Contextual 청크 예시]
[맥락] 삼성전자 분기보고서 중 사업 내용 소개 부분으로, 회사의 글로벌 조직과 주요 사업 부문별 제품 현황을 설명함.

II. 사업의 내용
 
1. 사업의 개요
 
당사는 본사를 거점으로 한국과 DX 부문 산하 해외 9개 지역총괄 및 DS 부문 산하 해외 5개
지역총괄의 생산ㆍ판매법인, SDC 및 Harman 산하 종속기업 등 310개의 종속기업으로 구성
된 글로벌 전자 기업입니다.
 
사업별로 보면, Set 사업은 DX(Device eXperience) 부문이 TV를 비롯하여 모니터, 냉장고,
세탁기, 에어컨, 스마트폰, 네트워크시스템, PC 등을 생산ㆍ판매하며, 부품 사업은
DS(Device Solutions) 부문에서 DRAM, NAND Flash, 모바일AP 등의 제품을 생산ㆍ판매
하고, SDC가 스마트폰용 OLED 패널 등을 생산ㆍ판매하고 있습니다.
또한, Harman에서는 디지털 콕핏(Digital Cockpit), 카오디오 등 전장제품과 포터블/사운드
바 스피커 등 컨슈머오디오 제품 등을 개발, 생산, 판매하고 있습니다.


`(3) 하이브리드 검색 및 평가`

Embedding + BM25 + Reranker 파이프라인을 구성하고 성능을 평가합니다.

In [21]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# 임베딩 모델 초기화
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 일반 청크 벡터 저장소 (비교용)
normal_vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="normal_chunks",
    persist_directory="./chroma_db"
)

# Contextual 청크 벡터 저장소
contextual_vectorstore = Chroma.from_documents(
    documents=contextual_chunks,
    embedding=embeddings,
    collection_name="contextual_chunks",
    persist_directory="./chroma_db"
)

print("벡터 저장소 생성 완료")
print(f"- 일반 청크: {normal_vectorstore._collection.count()}개")
print(f"- Contextual 청크: {contextual_vectorstore._collection.count()}개")

벡터 저장소 생성 완료
- 일반 청크: 274개
- Contextual 청크: 274개


In [22]:
# 테스트 쿼리
test_queries = [
    "삼성전자의 주요 제품은?",
    "삼성전자 주요 원재료는?",
    "삼성전자 TV 분야 현황은?",
]

# Retriever 생성
normal_retriever = normal_vectorstore.as_retriever(search_kwargs={"k": 3})
contextual_retriever = contextual_vectorstore.as_retriever(search_kwargs={"k": 3})

for query in test_queries:
    print(f"\n{'='*80}")
    print(f"쿼리: '{query}'")
    print("="*80)
    
    # 일반 검색
    normal_results = normal_retriever.invoke(query)
    print(f"\n[일반 검색 결과]")
    for i, doc in enumerate(normal_results):
        print(f"  {i+1}. {doc.page_content[:100]}...")
    
    # Contextual 검색
    contextual_results = contextual_retriever.invoke(query)
    print(f"\n[Contextual 검색 결과]")
    for i, doc in enumerate(contextual_results):
        print(f"  {i+1}. {doc.page_content[:100]}...")


쿼리: '삼성전자의 주요 제품은?'

[일반 검색 결과]
  1. 2. 주요 제품 및 서비스
 
가. 주요 제품 매출
 
당사는 TV, 냉장고, 세탁기, 에어컨, 스마트폰 등 완제품과 DRAM, NAND Flash, 모바일AP
등 반도체 부품 및...
  2. 3. 원재료 및 생산설비
 
가. 주요 원재료 현황 
 
당사의 주요 원재료로 DX 부문은 모바일AP 솔루션, 모바일용 메모리, Camera Module 등을
Qualcomm, M...
  3. 19.9% Qualcomm, MediaTek
디스플레이 패널
TVㆍ모니터용 화면표시장치
21,647
10.2% CSOT, AUO 등
모바일용 메모리
모바일용 메모리
19,930
9...

[Contextual 검색 결과]
  1. [맥락] 삼성전자 2026년 1분기 주요 제품 매출 현황과 가격 변동 추이를 상세히 설명하는 사업 내용 부분입니다.

2. 주요 제품 및 서비스
 
가. 주요 제품 매출
 
당사는...
  2. [맥락] 삼성전자 분기보고서 중 사업 내용 소개 부분으로, 회사의 글로벌 조직과 주요 사업 부문별 제품 현황을 설명함.

II. 사업의 내용
 
1. 사업의 개요
 
당사는 본사를...
  3. [맥락] 삼성전자 2026년 1분기 주요 제품별, 매출유형별, 지역별 매출 실적과 비교 현황을 다룬 부분입니다.

(1) 주요 제품별 매출실적 
 
(2) 매출유형별 매출실적 
 ...

쿼리: '삼성전자 주요 원재료는?'

[일반 검색 결과]
  1. 소 계
29,902
100.0% 　
Harman
SOC(System-On-Chip)
CPU
1,224
6.6% NVIDIA, INTEL 등
통신 모듈
차량 통신
1,899
10.2...
  2. 3. 원재료 및 생산설비
 
가. 주요 원재료 현황 
 
당사의 주요 원재료로 DX 부문은 모바일AP 솔루션, 모바일용 메모리, Camera Module 등을
Qualcomm, M...
  3. 나. 주요 원재료 가격 변동 추이
 
DX 부문의 주요 원재료인 모

## BM25 Retriever 설정

In [23]:
from langchain_community.retrievers import BM25Retriever
from kiwipiepy import Kiwi

# Kiwi 한국어 형태소 분석기 초기화
kiwi = Kiwi()

# 사용자 정의 단어 추가 (고유명사)
kiwi.add_user_word('반도체', 'NNP')
kiwi.add_user_word('삼성전자', 'NNP')

# 한국어 토크나이저 전처리 함수
def kiwi_preprocess_func(text):
    """Kiwi 형태소 분석기를 사용한 토큰화"""
    return [t.form for t in kiwi.tokenize(text)]

# 일반 BM25 Retriever (Kiwi 토크나이저 적용)
normal_bm25 = BM25Retriever.from_documents(
    documents=chunks,
    preprocess_func=kiwi_preprocess_func,
    k=3
)

# Contextual BM25 Retriever (Kiwi 토크나이저 적용)
contextual_bm25 = BM25Retriever.from_documents(
    documents=contextual_chunks,
    preprocess_func=kiwi_preprocess_func,
    k=3
)

print("BM25 Retriever 생성 완료 (Kiwi 토크나이저 적용)")

BM25 Retriever 생성 완료 (Kiwi 토크나이저 적용)


In [24]:
# 정확한 키워드가 필요한 쿼리
keyword_queries = [
    "Micro RGB",
    "메모리",
    "삼성전기",
    "삼성생명",
]

for query in keyword_queries:
    print(f"\n{'='*80}")
    print(f"쿼리: '{query}'")
    print("="*80)
    
    # 일반 BM25
    normal_results = normal_bm25.invoke(query)
    print(f"\n[일반 BM25]")
    for i, doc in enumerate(normal_results):
        print(f"  {i+1}. {doc.page_content[:100]}...")
    
    # Contextual BM25
    contextual_results = contextual_bm25.invoke(query)
    print(f"\n[Contextual BM25]")
    for i, doc in enumerate(contextual_results):
        print(f"  {i+1}. {doc.page_content[:100]}...")


쿼리: 'Micro RGB'

[일반 BM25]
  1. 3. 원재료 및 생산설비
 
가. 주요 원재료 현황 
 
당사의 주요 원재료로 DX 부문은 모바일AP 솔루션, 모바일용 메모리, Camera Module 등을
Qualcomm, M...
  2. 마. 주요 매출처
 
2026년 1분기 당사의 주요 매출처로는 Alphabet, Amazon, Apple, Hong Kong
Techtronics, Supreme Electroni...
  3. 사안별 상호 협의하에 일부 분담
Distributor
현지 거래처 직판 영업
개별계약조건
사안별 상호 협의하에 일부 분담
B2B 및
온라인
일반기업체, 삼성닷컴 등
개별계약조건
없...

[Contextual BM25]
  1. [맥락] 삼성전자 2026년 1분기 분기보고서 중 원재료 조달 현황과 주요 공급처를 설명하는 사업 내용 부분입니다.

3. 원재료 및 생산설비
 
가. 주요 원재료 현황 
 
당사...
  2. [맥락] 삼성전자 2026년 1분기 주요 매출처와 DS 부문 수주 현황, 주요 계약 내용 및 수주 총액을 소개하는 부분입니다.

마. 주요 매출처
 
2026년 1분기 당사의 주요...
  3. [맥락] 삼성전자 분기보고서 내 판매경로별 매출액 비중과 판매조건, 비용분담에 관한 국내외 판매 방법 및 조건 설명 부분입니다.

사안별 상호 협의하에 일부 분담
Distribut...

쿼리: '메모리'

[일반 BM25]
  1. 41,354
스마트폰 등
56,754
214,259
193,500
DS 부문
메모리
658,532,038
2,245,908,090
2,238,240,405
SDC
디스플레이 패널
...
  2. 19.9% Qualcomm, MediaTek
디스플레이 패널
TVㆍ모니터용 화면표시장치
21,647
10.2% CSOT, AUO 등
모바일용 메모리
모바일용 메모리
19,930
9...
  3. 부  문
품 목
제58기 1분기
가동 가능 시간
실제 가동 시간
가동률
DS 부문
메모리
21,6

## 하이브리드 검색 (Embedding + BM25)

In [25]:
from langchain_classic.retrievers import EnsembleRetriever

# 일반 하이브리드 Retriever
normal_hybrid = EnsembleRetriever(
    retrievers=[normal_retriever, normal_bm25],
    weights=[0.5, 0.5],  # Embedding과 BM25 동일 가중치
)

# Contextual 하이브리드 Retriever
contextual_hybrid = EnsembleRetriever(
    retrievers=[contextual_retriever, contextual_bm25],
    weights=[0.5, 0.5],
)

print("하이브리드 Retriever 생성 완료")

하이브리드 Retriever 생성 완료


In [27]:
# 다양한 유형의 쿼리
hybrid_queries = [
    "삼성전자 사업 실적은?",          
    "삼성전자 각 사업별 실적은?", 
]

for query in hybrid_queries:
    print(f"\n{'='*80}")
    print(f"쿼리: '{query}'")
    print("="*80)
    
    # 일반 하이브리드
    normal_results = normal_hybrid.invoke(query)
    print(f"\n[일반 하이브리드] 결과 {len(normal_results)}개")
    for i, doc in enumerate(normal_results[:2]):
        print(f"  {i+1}. {doc.page_content[:80]}...")
    
    # Contextual 하이브리드
    contextual_results = contextual_hybrid.invoke(query)
    print(f"\n[Contextual 하이브리드] 결과 {len(contextual_results)}개")
    for i, doc in enumerate(contextual_results[:2]):
        # 원본 내용 표시
        original = doc.metadata.get('original_content', doc.page_content[:80])
        print(f"  {i+1}. {original[:80]}...")


쿼리: '삼성전자 사업 실적은?'

[일반 하이브리드] 결과 5개
  1. 바 스피커 등 컨슈머오디오 제품 등을 개발, 생산, 판매하고 있습니다.
 
☞ 부문별 사업에 관한 자세한 사항은 '7. 기타 참고사항'의 '다....
  2. II. 사업의 내용
 
1. 사업의 개요
 
당사는 본사를 거점으로 한국과 DX 부문 산하 해외 9개 지역총괄 및 DS 부문 산하 해외 5개
지...

[Contextual 하이브리드] 결과 6개
  1. II. 사업의 내용
 
1. 사업의 개요
 
당사는 본사를 거점으로 한국과 DX 부문 산하 해외 9개 지역총괄 및 DS 부문 산하 해외 5개
지...
  2. (가동률)
 
당사 DX 부문의 2026년(제58기) 1분기 가동률은 생산능력 대비 생산실적으로 산출하였으
며, TV, 모니터 등은 82.2%,...

쿼리: '삼성전자 각 사업별 실적은?'

[일반 하이브리드] 결과 5개
  1. 바 스피커 등 컨슈머오디오 제품 등을 개발, 생산, 판매하고 있습니다.
 
☞ 부문별 사업에 관한 자세한 사항은 '7. 기타 참고사항'의 '다....
  2. 라. 생산설비 및 투자 현황 등 
 
(생산과 영업에 중요한 시설 및 설비 등)
 
당사는 수원사업장을 비롯하여 구미, 광주, 화성, 평택, 아...

[Contextual 하이브리드] 결과 6개
  1. 바 스피커 등 컨슈머오디오 제품 등을 개발, 생산, 판매하고 있습니다.
 
☞ 부문별 사업에 관한 자세한 사항은 '7. 기타 참고사항'의 '다....
  2. (1) 주요 제품별 매출실적 
 
(2) 매출유형별 매출실적 
 
(3) 주요 지역별 매출 현황 
(단위 : 억원)
구      분
제58기 1...


## Reranker 추가

하이브리드 검색 결과를 **Reranker로 재순위화**하여 최종 성능을 극대화합니다.

In [28]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# Cross-Encoder 모델 초기화
cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)

# Contextual Hybrid + Reranker
contextual_rerank_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=contextual_hybrid,
)

print("Reranker 설정 완료")

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 2989.15it/s]


Reranker 설정 완료


In [29]:
# 최종 검색 테스트
final_queries = [
    "삼성 TV는 사업 실적이 어떤가요?",          
    "메모리분야는 실적이 좋나요?", 
    "매출액은 어떤가요?",     # 혼합
]


for query in final_queries:
    print(f"\n{'='*80}")
    print(f"쿼리: '{query}'")
    print("="*80)
    
    # Contextual Hybrid + Reranker
    results = contextual_rerank_retriever.invoke(query, config={"callbacks": [langfuse_handler]})
    
    print(f"\n[Contextual Hybrid + Reranker] Top {len(results)}개")
    for i, doc in enumerate(results):
        original = doc.metadata.get('original_content', doc.page_content)
        context = doc.metadata.get('context', 'N/A')
        print(f"\n  [{i+1}] 맥락: {context}")
        print(f"      내용: {original[:100]}...")


쿼리: '삼성 TV는 사업 실적이 어떤가요?'

[Contextual Hybrid + Reranker] Top 3개

  [1] 맥락: 삼성전자 2026년 1분기 사업보고서 중 생산능력 대비 실제 생산 실적을 통한 각 부문별 가동률 현황을 설명하는 부분입니다.
      내용: (가동률)
 
당사 DX 부문의 2026년(제58기) 1분기 가동률은 생산능력 대비 생산실적으로 산출하였으
며, TV, 모니터 등은 82.2%, 스마트폰 등은 83.5%입니다. 
...

  [2] 맥락: 삼성전자 2026년 1분기 부문별 매출 실적과 전년 대비 성장률을 상세히 보고하는 매출 및 수주 현황 섹션 초반부입니다.
      내용: 4. 매출 및 수주상황
 
가. 매출실적
 
2026년(제58기) 1분기 매출은 133조 8,734억원으로 전년 동기 대비 69.2% 증가하였습니
다. 부문별로는 전년 동기 대비 ...

  [3] 맥락: 삼성전자 분기보고서 중 사업 내용 소개 부분으로, 회사의 글로벌 조직과 주요 사업 부문별 제품 현황을 설명함.
      내용: II. 사업의 내용
 
1. 사업의 개요
 
당사는 본사를 거점으로 한국과 DX 부문 산하 해외 9개 지역총괄 및 DS 부문 산하 해외 5개
지역총괄의 생산ㆍ판매법인, SDC 및 ...

쿼리: '메모리분야는 실적이 좋나요?'

[Contextual Hybrid + Reranker] Top 3개

  [1] 맥락: 삼성전자 2026년 1분기 부문별 주요 제품 생산실적과 생산 지역 현황을 상세히 기술한 부분입니다.
      내용: (생산실적)
 
2026년(제58기) 1분기 DX 부문의 TV, 모니터 등 생산실적은 11,354천대이며 멕시코, 베트
남, 브라질, 헝가리 등 세계 각 지역에서 생산되고 있습니다...

  [2] 맥락: 삼성전자 2026년 1분기 주요 제품별 생산실적 현황을 부문별 수치로 제시한 부분입니다.
      내용: 41,354
스마트폰 등
56,754
214,259
193,500
DS 부문
메모

In [30]:
from langchain_core.runnables import RunnablePassthrough

# 답변 생성용 LLM
answer_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# RAG 프롬프트
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 문서 기반 질의응답 전문가입니다.
주어진 문맥을 바탕으로 질문에 정확하게 답변하세요.
문맥에 없는 내용은 "문서에서 해당 정보를 찾을 수 없습니다."라고 답변하세요."""),
    ("user", """문맥:
{context}

질문: {question}

답변:""")
])

# 문서 포맷팅 함수
def format_docs(docs):
    formatted = []
    for doc in docs:
        # 원본 내용과 맥락 모두 포함
        original = doc.metadata.get('original_content', doc.page_content)
        context = doc.metadata.get('context', '')
        if context:
            formatted.append(f"[맥락: {context}]\n{original}")
        else:
            formatted.append(original)
    return "\n\n---\n\n".join(formatted)

# RAG 체인 구성
rag_chain = (
    {"context": contextual_rerank_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | answer_llm
    | StrOutputParser()
)

print("RAG 체인 구성 완료")

RAG 체인 구성 완료


In [31]:
# RAG 체인 실행
questions = [
    "삼성 TV는 사업 실적이 어떤가요?",          
    "메모리분야는 실적이 좋나요?", 
    "매출액은 어떤가요?",     # 혼합
]

for question in questions:
    print(f"\n{'='*80}")
    print(f"Q: {question}")
    print("-"*80)
    
    answer = rag_chain.invoke(question, config={"callbacks": [langfuse_handler]})
    print(f"A: {answer}")


Q: 삼성 TV는 사업 실적이 어떤가요?
--------------------------------------------------------------------------------
A: 삼성전자의 TV는 DX 부문에 속하며, 2026년 1분기 가동률은 생산능력 대비 82.2%입니다. 매출 측면에서는 DX 부문 전체가 2026년 1분기에 전년 동기 대비 1.8% 증가한 526,547억원의 매출을 기록하였으며, TV를 포함한 세트 제품군에 해당합니다. 따라서 삼성 TV 사업은 안정적인 가동률과 소폭의 매출 성장세를 보이고 있습니다.

Q: 메모리분야는 실적이 좋나요?
--------------------------------------------------------------------------------
A: 메모리 분야의 2026년 1분기 생산실적은 658,532백만개(1Gb 환산 기준)로 나타나고 있으며, DS 부문의 메모리 사업은 24시간 3교대 작업을 실시하고 있습니다. 다만, 가동률에 대한 구체적인 수치는 문서에 명시되어 있지 않습니다. 따라서 메모리 분야의 실적이 좋다고 단정하기에는 문서에서 해당 정보를 찾을 수 없습니다.

Q: 매출액은 어떤가요?
--------------------------------------------------------------------------------
A: 삼성전자의 2026년 1분기 주요 사업부문별 매출액은 다음과 같습니다.

- DX 부문: 526,547억원 (39.3%)
- DS 부문: 817,156억원 (61.0%)
- SDC: 66,935억원 (5.0%)
- Harman: 38,263억원 (2.9%)
- 기타(부문간 내부거래 제거 등): -110,167억원 (-8.2%)

총 매출액은 1,338,734억원이며, 각 부문별 매출액은 부문 간 내부거래를 포함하고 있습니다.


In [33]:
import random
import pandas as pd
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# ==========================================
# 1. 평가 데이터셋(QnA 쌍) 자동 생성
# ==========================================
print("데이터셋 생성을 위한 LLM 초기화 중...")
qa_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0) # gpt-4.1-mini 사용 가능

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 검색 엔진 평가용 데이터셋을 생성하는 전문가입니다. 주어진 문서의 일부(청크)를 읽고, 사용자가 이 내용을 찾기 위해 검색창에 입력할 법한 자연스럽고 명확한 '질문'을 1개만 생성하세요. 절대 질문 외의 부가 설명은 출력하지 마세요."),
    ("user", "문서 내용:\n{context}\n\n질문:")
])

qa_chain = qa_prompt | qa_llm | StrOutputParser()

# 전체 청크 중 20개를 샘플링하여 평가 (시간/비용 고려)
# * 더 정밀한 평가를 원하시면 sample_size를 늘리시면 됩니다.
sample_size = 20
random.seed(42)
sampled_chunks = random.sample(contextual_chunks, min(sample_size, len(contextual_chunks)))

qa_dataset = []
print(f"총 {len(sampled_chunks)}개의 평가용 질문 생성 중 (약 10~20초 소요)...\n")

for i, chunk in enumerate(sampled_chunks):
    # 정답 텍스트 추출
    gt_text = chunk.metadata.get('original_content', chunk.page_content)
    
    # LLM을 활용해 질문 생성
    question = qa_chain.invoke({"context": gt_text})
    
    qa_dataset.append({
        "query": question,
        "ground_truth_text": gt_text # 정답 비교를 위한 원본 텍스트
    })
    
    if (i + 1) % 5 == 0:
        print(f" - {i+1}개 완료...")

print("\n✅ 평가 데이터셋 생성 완료! 예시 질문:")
print(f"Q: {qa_dataset[0]['query']}")
print("="*80)

# ==========================================
# 2. 평가지표 (HitRate@K, MRR@K) 계산 함수
# ==========================================
def calculate_metrics(retriever, dataset, k=3):
    """
    HitRate: 정답 문서가 Top K 안에 들어있는 쿼리의 비율
    MRR (Mean Reciprocal Rank): 정답 문서가 위치한 순위의 역수(1/rank)의 평균
    """
    hits = 0
    rr_sum = 0.0
    
    for data in dataset:
        query = data["query"]
        gt_text = data["ground_truth_text"]
        
        try:
            # 검색 수행 (Reranker 등은 k를 자체적으로 설정하기도 하므로 [:k]로 한번 더 제한)
            results = retriever.invoke(query)[:k] 
        except Exception as e:
            continue
            
        rank = 0
        for i, doc in enumerate(results):
            # 일반 청크와 Contextual 청크의 원본 텍스트 위치가 다름을 보정하여 비교
            res_text = doc.metadata.get('original_content', doc.page_content)
            
            # 검색된 텍스트와 정답 텍스트가 일치하면 순위 기록
            if res_text == gt_text:
                rank = i + 1
                break
                
        if rank > 0:
            hits += 1
            rr_sum += 1.0 / rank
            
    total = len(dataset)
    hit_rate = hits / total if total > 0 else 0
    mrr = rr_sum / total if total > 0 else 0
    
    return {"HitRate@3": hit_rate, "MRR@3": mrr}

# ==========================================
# 3. 모든 Retriever 성능 비교 측정
# ==========================================
print("\n모든 Retriever 성능 평가 진행 중 (다소 시간이 소요됩니다)...\n")

# 비교할 전체 Retriever 딕셔너리 (이전 셀에서 생성한 변수들 재활용)
retrievers_to_compare = {
    "1. 일반 Embedding": normal_retriever,
    "2. 일반 BM25": normal_bm25,
    "3. 일반 Hybrid": normal_hybrid,
    "4. Contextual Embedding": contextual_retriever,
    "5. Contextual BM25": contextual_bm25,
    "6. Contextual Hybrid": contextual_hybrid,
    "7. Contextual Hybrid + Reranker": contextual_rerank_retriever,
}

evaluation_results = []

for name, retriever in retrievers_to_compare.items():
    metrics = calculate_metrics(retriever, qa_dataset, k=3)
    evaluation_results.append({
        "Retriever": name,
        "HitRate@3": f"{metrics['HitRate@3']:.2%}",
        "MRR@3": f"{metrics['MRR@3']:.3f}"
    })

# Pandas DataFrame으로 예쁘게 출력
df_results = pd.DataFrame(evaluation_results)
# MRR 기준으로 내림차순 정렬
df_results = df_results.sort_values(by="MRR@3", ascending=False).reset_index(drop=True)

# Jupyter 환경에서 예쁘게 표 렌더링
from IPython.display import display
display(df_results)

print("="*80)
print("💡 지표 해석:")
print("- HitRate@3: 사용자가 3위 이내의 검색 결과에서 정답을 찾을 확률입니다.")
print("- MRR@3: 정답이 상단에 노출될수록 점수가 높습니다. (1위=1.0, 2위=0.5, 3위=0.33)")

데이터셋 생성을 위한 LLM 초기화 중...
총 20개의 평가용 질문 생성 중 (약 10~20초 소요)...

 - 5개 완료...
 - 10개 완료...
 - 15개 완료...
 - 20개 완료...

✅ 평가 데이터셋 생성 완료! 예시 질문:
Q: 2026년 1분기 매출이 얼마나 증가했나요?

모든 Retriever 성능 평가 진행 중 (다소 시간이 소요됩니다)...



,Retriever,HitRate@3,MRR@3
0,4. Contextual Embedding,100.00%,0.908
1,7. Contextual Hybrid + Reranker,100.00%,0.900
2,6. Contextual Hybrid,100.00%,0.883
3,3. 일반 Hybrid,100.00%,0.875
4,1. 일반 Embedding,100.00%,0.850
5,5. Contextual BM25,85.00%,0.750
6,2. 일반 BM25,85.00%,0.717


💡 지표 해석:
- HitRate@3: 사용자가 3위 이내의 검색 결과에서 정답을 찾을 확률입니다.
- MRR@3: 정답이 상단에 노출될수록 점수가 높습니다. (1위=1.0, 2위=0.5, 3위=0.33)
